In [3]:
!pip install boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 2.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 103.1 MB/s eta 0:00:0000:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 8.2 MB/s eta 0:00:00


In [84]:
import pandas as pd
import boto3
from io import BytesIO
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq

In [5]:
s3 = boto3.client('s3',
                  endpoint_url = 'http://213.165.222.200:9000',
                  aws_access_key_id = 'minioadmin',
                  aws_secret_access_key = 'minioadmin')

In [85]:
SILVER_COLUMNS = {
    "id": "string",
    "period_cleaned": "datetime64[ns]",
    "retail_chain": "string",
    "category": "string",
    "category_2": "string",
    "supplier": "string",
    "brand": "string",
    "product_name": "string",
    "uni_product_name": "string",
    "grammage": "string",
    "flavor": "string",
    "sales_units": "double",
    "sales_rub": "double",
    "sales_tons": "double",
    "cost_rub": "double",
    "year": "int",
    "month": "string",
    "source_file": "string",
    "branch": "string",
    "region": "string",
    "city": "string",
    "address": "string",
    "distribution_center": "string",
    "trade_point": "string",
}


COLUMN_MAPPING = {
    # ID
    "_c0": "id",
    
    # Период
    "Период": "period",
    "период": "period",
    "Period": "period",
    
    # Сеть
    "Сеть": "retail_chain",
    "Сеть ": "retail_chain",
    "сеть": "retail_chain",
    "Retail": "retail_chain",
    
    # Категории
    "Категория": "category",
    "категория": "category",
    "Category": "category",
    
    "Категория 2": "category_2",
    "категория 2": "category_2",
    "Category 2": "category_2",
    'Тип основы': 'category_2',
    
    # Поставщик
    "Поставщик": "supplier",
    "поставщик": "supplier",
    "Supplier": "supplier",
    "Поставщики": "supplier",
    
    # Бренд
    "Бренд": "brand",
    "Бренды": "brand",
    "бренд": "brand",
    "Brand": "brand",
    
    # Наименование
    "Наименование": "product_name",
    "наименование": "product_name",
    "Product": "product_name",
    
    # УНИ Наименование
    "УНИ Наименование": "uni_product_name",
    "уни наименование": "uni_product_name",
    "UNI Name": "uni_product_name",
    
    # Граммовка
    "Граммовка": "grammage",
    "граммовка": "grammage",
    "Grammage": "grammage",
    
    # Вкус / Вкусы → единое поле
    "Вкус": "flavor",
    "Вкусы": "flavor",
    "вкус": "flavor",
    "вкусы": "flavor",
    "Flavor": "flavor",
    
    # Продажи (с пробелами и без)
    "Продажи, шт": "sales_units",
    "Продажи, шт ": "sales_units",
    "продажи, шт": "sales_units",
    "Sales Units": "sales_units",
    
    "Продажи, руб": "sales_rub",
    "Продажи, руб ": "sales_rub",
    "продажи, руб": "sales_rub",
    "Sales RUB": "sales_rub",
    
    "Продажи, тонн": "sales_tons",
    "Продажи, тонн ": "sales_tons",
    "продажи, тонн": "sales_tons",
    "Sales Tons": "sales_tons",
    
    # Себестоимость
    "Себест., руб": "cost_rub",
    "Себест., руб ": "cost_rub",
    "себест., руб": "cost_rub",
    "Cost RUB": "cost_rub",
    'Себест. Руб': "cost_rub",

    "Филиал ": "branch",           
    "Регион": "region",            
    "Город ": "city",               
    "Адрес": "address",             
    "РЦ": "distribution_center",    
    "ТТ": "trade_point",            
}

MONTH_MAPPING = {
    "january": 1, "february": 2, "march": 3,
    "april": 4, "may": 5, "june": 6,
    "july": 7, "august": 8, "september": 9,
    "ceptember": 9,
    "october": 10, "november": 11, "december": 12
}

In [27]:
objects = s3.list_objects(Bucket = 'data')['Contents']
print(objects)

[{'Key': '2024/ceptember/okey', 'LastModified': datetime.datetime(2026, 4, 2, 14, 33, 10, 829000, tzinfo=tzlocal()), 'ETag': '"ea0bc05e397405bb3934489e34ee781c"', 'Size': 65074, 'StorageClass': 'STANDARD', 'Owner': {'DisplayName': 'minio', 'ID': '02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4'}}, {'Key': '2024/ceptember/perekrestok', 'LastModified': datetime.datetime(2026, 4, 2, 14, 33, 17, 716000, tzinfo=tzlocal()), 'ETag': '"d37a69ad4942ef6793896b68f479ab01"', 'Size': 98917, 'StorageClass': 'STANDARD', 'Owner': {'DisplayName': 'minio', 'ID': '02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4'}}, {'Key': '2024/ceptember/pyaterochka', 'LastModified': datetime.datetime(2026, 4, 2, 14, 33, 17, 729000, tzinfo=tzlocal()), 'ETag': '"84c4852e50425269eca2ab0dde911935"', 'Size': 64818, 'StorageClass': 'STANDARD', 'Owner': {'DisplayName': 'minio', 'ID': '02d6176db174dc93cb1b899f7c6078f08654445fe8cf1b6ce98d8855f66bdbf4'}}]


In [83]:
for file in objects:
    file_name = file['Key']
    date_part = file_name.split('/')

    obj = s3.get_object(Bucket="data", Key=file_name)

    df = pd.read_csv(BytesIO(obj['Body'].read()))
    print(df.dtypes)
    month_parse = date_part[1]


    for i in df.columns:
        if i in COLUMN_MAPPING.keys():
            new_name = COLUMN_MAPPING.get(i)
            df = df.rename(columns = {i: new_name})
    
    df['year'] = date_part[0]
    df['month'] = str(MONTH_MAPPING.get(month_parse))
    df['retail_chain'] = date_part[2]
    df['source_file'] = file_name
    df['period_cleaned'] = pd.to_datetime(df['year'] + '-' + df['month'] + '-01', format='%Y-%m-%d')
    
    missing_columns = set(SILVER_COLUMNS.keys()) - set(df.columns)
    
    for i in missing_columns:
        df[i] = np.nan
    
    for col in df.columns:
        if col in SILVER_COLUMNS.keys():
            dtype = SILVER_COLUMNS.get(col)
            df[col] = df[col].astype(dtype)

    df = df[SILVER_COLUMNS.keys()]
    print(df.dtypes)
    display(df.head())


    

Unnamed: 0            int64
Период               object
Сеть                 object
Категория            object
Категория 2          object
Поставщик            object
Бренд                object
Наименование         object
УНИ Наименование     object
Граммовка             int64
Вкус                 object
Продажи, шт           int64
Продажи, руб        float64
Продажи, тонн       float64
Себест., руб        float64
dtype: object
id                     string[python]
period_cleaned         datetime64[ns]
retail_chain           string[python]
category               string[python]
category_2             string[python]
supplier               string[python]
brand                  string[python]
product_name           string[python]
uni_product_name       string[python]
grammage               string[python]
flavor                 string[python]
sales_units                   float64
sales_rub                     float64
sales_tons                    float64
cost_rub                      floa

,id,period_cleaned,retail_chain,category,category_2,supplier,brand,product_name,uni_product_name,grammage,...,cost_rub,year,month,source_file,branch,region,city,address,distribution_center,trade_point
0,<NA>,2024-09-01,okey,Чипсы картофельные обычные,ЧСК обычные,Seykar gida san. VE. TIC. LTD.,PATTES,Чипсы картофельные Pattes классические рифлены...,PATTES Оригинальные 100г,100,...,40009.54,2024,9,2024/ceptember/okey,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,<NA>,2024-09-01,okey,Чипсы картофельные обычные,ЧСК обычные,Seykar gida san. VE. TIC. LTD.,PATTES,Чипсы картофельные Pattes со вкусом зелени и й...,PATTES Йогурт и травы 100г,100,...,55629.67,2024,9,2024/ceptember/okey,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,<NA>,2024-09-01,okey,Чипсы картофельные обычные,ЧСК обычные,Seykar gida san. VE. TIC. LTD.,PATTES,Чипсы картофельные Pattes со вкусом кетчупа 10...,PATTES Кетчуп 100г,100,...,44873.65,2024,9,2024/ceptember/okey,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,<NA>,2024-09-01,okey,Чипсы картофельные обычные,ЧСК обычные,Seykar gida san. VE. TIC. LTD.,PATTES,Чипсы картофельные Pattes со вкусом красного п...,PATTES Чили 100г,100,...,58027.42,2024,9,2024/ceptember/okey,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,<NA>,2024-09-01,okey,Чипсы картофельные обычные,ЧСК обычные,Seykar gida san. VE. TIC. LTD.,PATTES,Чипсы картофельные Pattes со вкусом уксуса и л...,PATTES Уксус и лайм 100г,100,...,43640.53,2024,9,2024/ceptember/okey,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


Unnamed: 0            int64
Период               object
Сеть                 object
Категория            object
Категория 2          object
Поставщик            object
Бренд                object
Наименование         object
УНИ Наименование     object
Граммовка             int64
Вкусы                object
Продажи, шт           int64
Продажи, руб        float64
Продажи, тонн       float64
Себест., руб        float64
dtype: object
id                     string[python]
period_cleaned         datetime64[ns]
retail_chain           string[python]
category               string[python]
category_2             string[python]
supplier               string[python]
brand                  string[python]
product_name           string[python]
uni_product_name       string[python]
grammage               string[python]
flavor                 string[python]
sales_units                   float64
sales_rub                     float64
sales_tons                    float64
cost_rub                      floa

,id,period_cleaned,retail_chain,category,category_2,supplier,brand,product_name,uni_product_name,grammage,...,cost_rub,year,month,source_file,branch,region,city,address,distribution_center,trade_point
0,<NA>,2024-09-01,perekrestok,Чипсы картофельные обычные,ЧСК обычные,"АО ""ТРАНСАТЛАНТИК ИНТЕРНЕЙШНЛ""",Savoursmiths,SAVOUR.Чип.кар.вкус.ит.сыр/пор.руб.150г,Savoursmiths Итальянский сыр и порто руби 150г,150,...,58307.640,2024,9,2024/ceptember/perekrestok,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,<NA>,2024-09-01,perekrestok,Чипсы картофельные обычные,ЧСК обычные,"АО ""ТРАНСАТЛАНТИК ИНТЕРНЕЙШНЛ""",Savoursmiths,SAVOUR.Чип.карт.сыр.чеддр/лук-шалот 150г,Savoursmiths Сыр.чеддр и лук-шалот 150г,150,...,62916.060,2024,9,2024/ceptember/perekrestok,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,<NA>,2024-09-01,perekrestok,Чипсы картофельные обычные,ЧСК обычные,"АО ""ТРАНСАТЛАНТИК ИНТЕРНЕЙШНЛ""",Savoursmiths,SAVOURSM.Чипсы карт.с сол.пуст.150г,Savoursmiths Соль пустыни 150г,150,...,23510.028,2024,9,2024/ceptember/perekrestok,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,<NA>,2024-09-01,perekrestok,Чипсы картофельные обычные,ЧСК обычные,Seykar gida san. VE. TIC. LTD.,PATTES,PATTES Чипсы КЛАССИЧЕСКИЕ картофель.100г,PATTES Оригинальные 100г,100,...,7181.700,2024,9,2024/ceptember/perekrestok,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,<NA>,2024-09-01,perekrestok,Чипсы картофельные обычные,ЧСК обычные,Seykar gida san. VE. TIC. LTD.,PATTES,PATTES Чипсы со вк.йогур/трав карт.100г,PATTES Йогурт и травы 100г,100,...,604.800,2024,9,2024/ceptember/perekrestok,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


Unnamed: 0            int64
Период               object
Сеть                 object
Категория            object
Тип основы           object
Поставщики           object
Бренды               object
Наименование         object
УНИ Наименование     object
Граммовка             int64
Вкусы                object
Продажи, шт           int64
Продажи, руб        float64
Продажи, тонн       float64
Себест., руб        float64
dtype: object
id                     string[python]
period_cleaned         datetime64[ns]
retail_chain           string[python]
category               string[python]
category_2             string[python]
supplier               string[python]
brand                  string[python]
product_name           string[python]
uni_product_name       string[python]
grammage               string[python]
flavor                 string[python]
sales_units                   float64
sales_rub                     float64
sales_tons                    float64
cost_rub                      floa

,id,period_cleaned,retail_chain,category,category_2,supplier,brand,product_name,uni_product_name,grammage,...,cost_rub,year,month,source_file,branch,region,city,address,distribution_center,trade_point
0,<NA>,2024-09-01,pyaterochka,Чипсы картофельные обычные,ЧСК в пачке-чаше,Маревен Фуд Сэнтрал ООО,Big Bon,BIG BON SN.B.Чип.кар.вк.Сыр с хр.бек.70г,BIG BON Snack Box Сыр с хруст. Беконом 70г,70,...,44142.696,2024,9,2024/ceptember/pyaterochka,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,<NA>,2024-09-01,pyaterochka,Чипсы картофельные обычные,ЧСК в пачке-чаше,Маревен Фуд Сэнтрал ООО,Big Bon,BIG BON SN.B.Чипсы карт.вк.Коп.папр.70г,BIG BON Snack Box Копченая паприка 70г,70,...,42817.152,2024,9,2024/ceptember/pyaterochka,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,<NA>,2024-09-01,pyaterochka,Чипсы картофельные обычные,ЧСК обычные,Импортлогистик ООО,San Carlo,S.CAR.Чипсы картоф.лайм/роз.пер.150г,San Carlo Лайм и розовый перец 150 г,150,...,840.312,2024,9,2024/ceptember/pyaterochka,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,<NA>,2024-09-01,pyaterochka,Чипсы картофельные обычные,ЧСК обычные,Импортлогистик ООО,San Carlo,S.CAR.Чипсы картоф.паприка 150г,San Carlo Паприка 150 г,150,...,164.676,2024,9,2024/ceptember/pyaterochka,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,<NA>,2024-09-01,pyaterochka,Чипсы картофельные обычные,ЧСК обычные,Импортлогистик ООО,San Carlo,S.CAR.Чипсы картоф.томат 150г,San Carlo Томат 150 г,150,...,166.704,2024,9,2024/ceptember/pyaterochka,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
